In [ ]:
"import sys\n","sys.path.insert(0, '../src')\n","\n","import torch\n","from torch.utils.data import DataLoader\n","import pytorch_lightning as pl\n","from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping\n","from pytorch_lightning.loggers import TensorBoardLogger\n","\n","from config import load_config\n","from dataset_coco import COCODetectionDataset\n","from models import DETR\n","from lightning_module import DetectionModule\n"

In [ ]:
"config = load_config('../configs/config_coco_detr_person.json')\n","\n","DATA_ROOT = config['data']['root_dir']\n","TRAIN_SPLIT = config['data']['split']\n","VAL_SPLIT = config['data']['val_split']\n","TEST_SPLIT = config['data']['test_split']\n","TARGET_SIZE = tuple(config['data']['target_size'])\n","BATCH_SIZE = config['data']['batch_size']\n","NUM_WORKERS = config['data']['num_workers']\n","MIN_AREA = config['data']['min_area']\n","FILTER_CATEGORIES = config['data']['filter_categories']\n","NUM_CLASSES = config['model']['num_classes']\n","\n","MAX_EPOCHS = config['training']['max_epochs']\n","LEARNING_RATE = config['training']['learning_rate']\n","WEIGHT_DECAY = config['training']['weight_decay']\n","OPTIMIZER = config['training']['optimizer']\n","\n","NUM_QUERIES = config['model']['num_queries']\n","EMB_DIM = config['model']['emb_dim']\n","NHEAD = config['model']['nhead']\n","ENC_LAYERS = config['model']['enc_layers']\n","DEC_LAYERS = config['model']['dec_layers']\n","PRETRAINED = config['model']['pretrained']\n","\n","print(f\"Model: DETR\")\n","print(f\"Classes: {NUM_CLASSES} ({', '.join(FILTER_CATEGORIES)})\")\n","print(f\"Num queries: {NUM_QUERIES}\")\n","print(f\"Encoder layers: {ENC_LAYERS}, Decoder layers: {DEC_LAYERS}\")\n"

In [ ]:
"def collate_fn(batch):\n","    images = torch.stack([item[0] for item in batch])\n","    boxes = [item[1] for item in batch]\n","    labels = [item[2] for item in batch]\n","    return images, boxes, labels\n","\n","train_dataset = COCODetectionDataset(\n","    root_dir=DATA_ROOT,\n","    split=TRAIN_SPLIT,\n","    target_size=TARGET_SIZE,\n","    min_area=MIN_AREA,\n","    filter_categories=FILTER_CATEGORIES\n",")\n","\n","val_dataset = COCODetectionDataset(\n","    root_dir=DATA_ROOT,\n","    split=VAL_SPLIT,\n","    target_size=TARGET_SIZE,\n","    min_area=MIN_AREA,\n","    filter_categories=FILTER_CATEGORIES\n",")\n","\n","test_dataset = COCODetectionDataset(\n","    root_dir=DATA_ROOT,\n","    split=TEST_SPLIT,\n","    target_size=TARGET_SIZE,\n","    min_area=MIN_AREA,\n","    filter_categories=FILTER_CATEGORIES\n",")\n","\n","print(f'Train samples: {len(train_dataset)}')\n","print(f'Val samples: {len(val_dataset)}')\n","print(f'Test samples: {len(test_dataset)}')\n"

In [ ]:
"train_loader = DataLoader(\n","    train_dataset,\n","    batch_size=BATCH_SIZE,\n","    shuffle=True,\n","    num_workers=NUM_WORKERS,\n","    collate_fn=collate_fn,\n","    pin_memory=True\n",")\n","\n","val_loader = DataLoader(\n","    val_dataset,\n","    batch_size=BATCH_SIZE,\n","    shuffle=False,\n","    num_workers=NUM_WORKERS,\n","    collate_fn=collate_fn,\n","    pin_memory=True\n",")\n","\n","test_loader = DataLoader(\n","    test_dataset,\n","    batch_size=BATCH_SIZE,\n","    shuffle=False,\n","    num_workers=NUM_WORKERS,\n","    collate_fn=collate_fn,\n","    pin_memory=True\n",")\n","\n","print(f'Train batches: {len(train_loader)}')\n","print(f'Val batches: {len(val_loader)}')\n","print(f'Test batches: {len(test_loader)}')\n"

In [ ]:
"model = DETR(\n","    num_classes=NUM_CLASSES,\n","    emb_dim=EMB_DIM,\n","    num_queries=NUM_QUERIES,\n","    nhead=NHEAD,\n","    enc_layers=ENC_LAYERS,\n","    dec_layers=DEC_LAYERS,\n","    pretrained=PRETRAINED\n",")\n","\n","lightning_module = DetectionModule(\n","    model=model,\n","    num_classes=NUM_CLASSES,\n","    learning_rate=LEARNING_RATE,\n","    weight_decay=WEIGHT_DECAY,\n","    optimizer=OPTIMIZER\n",")\n","\n","print(f\"Model initialized\")\n","print(f\"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M\")\n"

In [ ]:
"checkpoint_callback = ModelCheckpoint(\n","    dirpath=config['training']['checkpoint_dirpath'],\n","    filename=config['training']['checkpoint_filename'],\n","    monitor=config['training']['checkpoint_monitor'],\n","    mode=config['training']['checkpoint_mode'],\n","    save_top_k=config['training']['checkpoint_save_top_k'],\n","    verbose=True\n",")\n","\n","early_stopping_callback = EarlyStopping(\n","    monitor=config['training']['checkpoint_monitor'],\n","    patience=config['training']['early_stopping_patience'],\n","    mode=config['training']['checkpoint_mode'],\n","    verbose=True\n",")\n","\n","logger = TensorBoardLogger(\n","    save_dir=config['training']['log_dir'],\n","    name=config['training']['experiment_name']\n",")\n","\n","trainer = pl.Trainer(\n","    max_epochs=MAX_EPOCHS,\n","    accelerator=config['hardware']['accelerator'],\n","    devices=config['hardware']['devices'],\n","    callbacks=[checkpoint_callback, early_stopping_callback],\n","    logger=logger,\n","    log_every_n_steps=config['training']['log_every_n_steps'],\n","    deterministic=False\n",")\n","\n","print(\"Trainer configured\")\n","print(f\"Max epochs: {MAX_EPOCHS}\")\n","print(f\"Device: {config['hardware']['accelerator']}\")\n"

In [ ]:
"trainer.fit(lightning_module, train_loader, val_loader)\n"